# <font color="#418FDE" size="6.5" uppercase>**Sparse Robust Models**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Select sparse linear methods when only a subset of features is expected to matter. 
- Apply Bayesian linear models to estimate coefficients with uncertainty-aware regularization. 
- Use robust regression methods when outliers or nonstandard error patterns affect least squares. 


## **1. Sparse Linear Methods**

### **1.1. Lasso Feature Selection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_01_01.jpg?v=1783841136" width="250">



>* Lasso keeps important features, removes weak ones.
>* Simpler models explain better and generalize well.

>* Lasso sets some coefficients exactly to zero.
>* Keeps strongest predictors for simpler decisions.

>* Lasso selections depend on data conditions.
>* Sparse models aid simplicity, but need caution.



In [ ]:
#@title Python Code - Lasso Feature Selection

# Lasso can remove weak input features.
# This example uses a tiny synthetic dataset.
# We compare ordinary fit and sparse fit.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create small feature matrix.
X = rng.normal(size=(60, 8))

# Define only two true effects.
true_w = np.array([3.0, 0.0, -2.5, 0.0, 0.0, 0.0, 0.0, 0.0])

y = X @ true_w + rng.normal(0, 0.8, 60)

# Standardize columns for fair shrinkage.
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_std = np.where(X_std == 0, 1.0, X_std)

# Build standardized design matrix.
Xs = (X - X_mean) / X_std

# Check shapes before fitting.
assert Xs.shape == (60, 8)
assert y.shape == (60,)

# Solve ordinary least squares.
ols_w = np.linalg.lstsq(Xs, y, rcond=None)[0]

# Define soft thresholding helper.
def soft_threshold(value, penalty):
    return np.sign(value) * max(abs(value) - penalty, 0.0)

# Fit lasso using coordinate descent.
lasso_w = np.zeros(Xs.shape[1])
alpha = 0.7

# Run a short stable optimization.
for _ in range(120):
    for j in range(Xs.shape[1]):
        residual = y - Xs @ lasso_w + Xs[:, j] * lasso_w[j]
        rho = (Xs[:, j] * residual).mean()
        z = (Xs[:, j] ** 2).mean()
        lasso_w[j] = soft_threshold(rho, alpha) / z

# Count selected nonzero features.
selected = np.abs(lasso_w) > 1e-6
count_selected = int(selected.sum())

# Print a short teaching summary.
print("True important features: 0 and 2")
print("OLS nonzero coefficients:", int((np.abs(ols_w) > 1e-6).sum()))
print("Lasso selected features:", np.where(selected)[0].tolist())
print("Lasso selected count:", count_selected)

# Plot coefficient comparison clearly.
positions = np.arange(len(true_w))
width = 0.25

# Draw true, OLS, and lasso bars.
plt.figure(figsize=(8, 4))
plt.bar(positions - width, true_w, width, label="True")
plt.bar(positions, ols_w, width, label="OLS")
plt.bar(positions + width, lasso_w, width, label="Lasso")

# Add labels for readability.
plt.xticks(positions, [f"x{i}" for i in positions])
plt.ylabel("Coefficient value")
plt.title("Lasso keeps only stronger signals")
plt.legend()

# Show the final teaching plot.
plt.tight_layout()
plt.show()



### **1.2. Lasso Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_01_02.jpg?v=1783841128" width="250">



>* Lasso shrinks unimportant coefficients to zero.
>* It balances accuracy, simplicity, and interpretability.

>* Lasso selects features while estimating coefficients.
>* It stabilizes high-dimensional models by shrinking weak predictors.

>* Correlated features make lasso selections less definitive.
>* Careful tuning yields sparse, practical predictive models.



In [ ]:
#@title Python Code - Lasso Regression

# This example shows sparse linear regression.
# Lasso can remove weak input features.
# We build it with coordinate descent.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create small synthetic feature data.
n_samples, n_features = 80, 8
X = rng.normal(size=(n_samples, n_features))

# Define only two true coefficients.
true_coef = np.array([3.0, 0.0, 0.0, -2.5, 0.0, 0.0, 0.0, 0.0])
y = X @ true_coef + rng.normal(scale=0.8, size=n_samples)

# Standardize features for fair shrinkage.
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_std = np.where(X_std == 0, 1.0, X_std)
X_scaled = (X - X_mean) / X_std

# Center the target values.
y_mean = y.mean()
y_centered = y - y_mean

# Check shapes before fitting.
assert X_scaled.shape == (n_samples, n_features)
assert y_centered.shape == (n_samples,)

# Define soft thresholding helper.
def soft_threshold(value, penalty):
    if value > penalty:
        return value - penalty
    if value < -penalty:
        return value + penalty
    return 0.0

# Fit lasso using coordinate descent.
def fit_lasso_cd(X_data, y_data, alpha, steps=100):
    rows, cols = X_data.shape
    coef = np.zeros(cols)
    for _ in range(steps):

        for j in range(cols):
            residual = y_data - X_data @ coef + X_data[:, j] * coef[j]
            rho = X_data[:, j] @ residual
            scale = (X_data[:, j] ** 2).sum()
            coef[j] = soft_threshold(rho, alpha) / scale

    return coef

# Fit ordinary least squares baseline.
ols_coef = np.linalg.lstsq(X_scaled, y_centered, rcond=None)[0]

# Fit lasso with moderate penalty.
alpha = 18.0
lasso_coef = fit_lasso_cd(X_scaled, y_centered, alpha, steps=120)

# Count coefficients near zero.
near_zero = np.sum(np.isclose(lasso_coef, 0.0, atol=0.08))

# Print a short teaching summary.
print("True nonzero features:", [0, 3])
print("OLS nonzero count:", int(np.sum(np.abs(ols_coef) > 0.08)))
print("Lasso near-zero count:", int(near_zero))
print("Lasso coefficients:", np.round(lasso_coef, 2))

# Plot coefficient shrinkage comparison.
positions = np.arange(n_features)
plt.figure(figsize=(8, 4))
plt.axhline(0.0, color="gray", linewidth=1)
plt.plot(positions, ols_coef, "o-", label="OLS")
plt.plot(positions, lasso_coef, "s-", label="Lasso")

# Add labels for readability.
plt.xticks(positions, [f"x{i}" for i in positions])
plt.ylabel("Coefficient value")
plt.title("Lasso shrinks weak features toward zero")
plt.legend()
plt.tight_layout()
plt.show()



### **1.3. Choosing Sparse Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_01_03.jpg?v=1783841131" width="250">



>* Few features often contain most signal.
>* Sparse models improve simplicity and generalization.

>* Correlated features may add little unique value.
>* Choose stable, sensible features across datasets.

>* Sparse choices depend on model purpose.
>* Selected features are not automatic truths.



## **2. Bayesian Linear Models**

### **2.1. Bayesian Coefficient Uncertainty**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_02_01.jpg?v=1783841143" width="250">



>* Coefficients are distributions, not fixed values.
>* Uncertainty varies with data quality and overlap.

>* Uncertainty reveals whether effects are reliable.
>* Wide coefficient ranges signal caution in decisions.

>* Weak evidence shrinks coefficients toward zero.
>* Strong predictors stand out, improving stability.



In [ ]:
#@title Python Code - Bayesian Coefficient Uncertainty

# Bayesian coefficients can express uncertainty clearly.
# This example compares certainty across two predictors.
# Small noisy data makes uncertainty easy to see.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create one informative predictor.
x1 = np.linspace(0.0, 10.0, 30)

# Create one weak noisy predictor.
x2 = x1 + rng.normal(0.0, 0.3, 30)

# Build the design matrix.
X = np.column_stack([np.ones(30), x1, x2])

# Generate targets with noise.
y = 3.0 + 2.0 * x1 + rng.normal(0.0, 2.0, 30)

# Validate matrix dimensions first.
assert X.shape == (30, 3)
assert y.shape == (30,)

# Set a simple Gaussian prior.
alpha = 1.0
beta = 1.0 / (2.0 ** 2)

# Compute posterior covariance matrix.
SN = np.linalg.inv(alpha * np.eye(3) + beta * X.T @ X)

# Compute posterior mean vector.
mN = beta * SN @ X.T @ y

# Extract coefficient uncertainties.
stds = np.sqrt(np.diag(SN))
labels = ["intercept", "x1 strong", "x2 uncertain"]

# Print a short interpretation.
for name, mean, std in zip(labels, mN, stds):
    print(f"{name}: mean={mean:.2f}, std={std:.2f}")

# Plot means with uncertainty bars.
plt.figure(figsize=(6, 4))
plt.errorbar(labels, mN, yerr=2 * stds, fmt="o", capsize=5)

# Add a helpful title.
plt.title("Bayesian coefficient uncertainty")
plt.ylabel("Posterior mean ± 2 std")

# Keep the layout tidy.
plt.tight_layout()
plt.show()



### **2.2. Bayesian Regression Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_02_02.jpg?v=1783841138" width="250">



>* Coefficients start as uncertain, guided by priors.
>* Data update beliefs into useful posteriors.

>* Bayesian regression updates coefficient ranges with data.
>* Posterior width shows confidence in learned effects.

>* Quantifies uncertainty in coefficients and predictions.
>* Priors stabilize estimates through regularization.



In [ ]:
#@title Python Code - Bayesian Regression Basics

# Bayesian priors shrink uncertain coefficient estimates.
# This example compares ordinary and Bayesian regression.
# Small data makes uncertainty effects easier.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create one useful predictor feature.
x1 = np.linspace(0.0, 10.0, 20)

# Create one mostly irrelevant feature.
x2 = rng.normal(0.0, 3.0, 20)

# Build the design matrix.
X = np.column_stack([np.ones(20), x1, x2])

# Validate matrix dimensions safely.
assert X.shape == (20, 3)

# Generate noisy target values.
y = 5.0 + 2.0 * x1 + rng.normal(0.0, 4.0, 20)

# Compute ordinary least squares coefficients.
ols_beta = np.linalg.solve(X.T @ X, X.T @ y)

# Set a Gaussian prior strength.
alpha = 2.0

# Compute Bayesian posterior mean coefficients.
post_beta = np.linalg.solve(X.T @ X + alpha * np.eye(3), X.T @ y)

# Estimate residual noise from Bayesian fit.
resid = y - X @ post_beta
sigma2 = float((resid @ resid) / (len(y) - X.shape[1]))

# Compute posterior covariance matrix.
post_cov = sigma2 * np.linalg.inv(X.T @ X + alpha * np.eye(3))

# Summarize coefficient uncertainty briefly.
post_sd = np.sqrt(np.diag(post_cov))

# Print a compact teaching summary.
print("Feature order: intercept, x1, x2")
print("OLS coefficients:", np.round(ols_beta, 2))
print("Bayesian mean:", np.round(post_beta, 2))
print("Posterior SDs:", np.round(post_sd, 2))

# Create a prediction grid.
x1_grid = np.linspace(0.0, 10.0, 100)

# Hold the irrelevant feature fixed.
X_grid = np.column_stack([np.ones(100), x1_grid, np.zeros(100)])

# Compute Bayesian mean predictions.
y_grid = X_grid @ post_beta

# Plot data and Bayesian fit.
plt.figure(figsize=(7, 4))
plt.scatter(x1, y, color="steelblue", label="Observed data")
plt.plot(x1_grid, y_grid, color="darkred", label="Bayesian fit")
plt.xlabel("Useful predictor x1")
plt.ylabel("Target y")

# Add a short explanatory title.
plt.title("Bayesian regression shrinks uncertain coefficients")
plt.legend()
plt.tight_layout()
plt.show()



### **2.3. Uncertainty Aware Regularization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_02_03.jpg?v=1783841145" width="250">



>* Bayesian shrinkage reflects coefficient uncertainty.
>* Balances priors, data, and noisy predictors.

>* Data and priors shrink features differently.
>* Correlated noisy predictors get cautious estimates.

>* Uncertainty improves coefficient based decision making.
>* Priors encode assumptions for honest estimates.



In [ ]:
#@title Python Code - Uncertainty Aware Regularization

# Bayesian priors shrink uncertain coefficients gently.
# Posterior spread shows confidence in effects.
# This example uses closed form updates.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Build correlated predictors and noise.
n_samples = 40
x1 = rng.normal(size=n_samples)
x2 = x1 + rng.normal(scale=0.25, size=n_samples)

# Create a design matrix.
X = np.column_stack((np.ones(n_samples), x1, x2))
true_beta = np.array([1.0, 2.0, 0.0])
noise = rng.normal(scale=1.2, size=n_samples)

y = X @ true_beta + noise

# Check matrix dimensions safely.
assert X.shape == (n_samples, 3)
assert y.shape == (n_samples,)

# Set Bayesian ridge style priors.
alpha = 1.0
sigma2 = 1.2 ** 2

# Compute posterior covariance matrix.
precision = (X.T @ X) / sigma2
precision += alpha * np.eye(X.shape[1])
posterior_cov = np.linalg.inv(precision)

# Compute posterior mean coefficients.
posterior_mean = posterior_cov @ (X.T @ y / sigma2)
posterior_sd = np.sqrt(np.diag(posterior_cov))

# Compare with ordinary least squares.
ols_beta = np.linalg.lstsq(X, y, rcond=None)[0]

# Print a short teaching summary.
print("Posterior means:", np.round(posterior_mean, 2))
print("Posterior std devs:", np.round(posterior_sd, 2))
print("OLS coefficients:", np.round(ols_beta, 2))
print("x2 stays uncertain, so shrinkage remains cautious.")

# Plot means with uncertainty bars.
labels = ["intercept", "x1", "x2"]
positions = np.arange(len(labels))
plt.figure(figsize=(7, 4))

# Show posterior means and intervals.
plt.errorbar(
    positions, posterior_mean, yerr=2 * posterior_sd,
    fmt="o", capsize=5, label="Bayesian mean ± 2 sd"
)

# Add true and OLS references.
plt.scatter(positions, true_beta, marker="x", s=80, label="true")
plt.scatter(positions, ols_beta, marker="s", s=50, label="OLS")
plt.xticks(positions, labels)

# Finish the single teaching plot.
plt.axhline(0.0, color="gray", linewidth=1)
plt.title("Uncertainty aware regularization")
plt.ylabel("coefficient value")
plt.legend()

plt.tight_layout()
plt.show()



## **3. Robust Regression Methods**

### **3.1. Huber Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_03_01.jpg?v=1783841146" width="250">



>* Huber regression reduces outlier influence.
>* Keeps least squares behavior for small errors.

>* Small errors get full influence.
>* Large errors are downweighted across applications.

>* Threshold choice balances robustness and efficiency.
>* Huber suits linear data with unreliable assumptions.



### **3.2. Robust Loss Functions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_03_02.jpg?v=1783841151" width="250">



>* Reduces outliers’ influence during regression fitting.
>* Softens large-error penalties, preserving overall patterns.

>* Near residuals count more than distant ones.
>* Large errors are softened, limiting outlier influence.

>* Robust loss balances stability with sensitivity.
>* Outliers still need careful investigation.



In [ ]:
#@title Python Code - Robust Loss Functions

# Robust losses reduce outlier influence.
# This example compares two fitting rules.
# We use only numpy and matplotlib.

import numpy as np
import matplotlib.pyplot as plt

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create simple linear data.
x = np.linspace(0.0, 10.0, 30)
y = 2.0 * x + 3.0
noise = rng.normal(0.0, 1.0, size=x.size)

# Add ordinary noise first.
y_noisy = y + noise

# Inject a few strong outliers.
outlier_idx = np.array([4, 18, 25])
y_noisy[outlier_idx] += np.array([14.0, -16.0, 12.0])

# Check matching input shapes.
assert x.shape == y_noisy.shape

# Build the design matrix.
X = np.column_stack([x, np.ones_like(x)])

# Fit ordinary least squares.
ols_coef = np.linalg.lstsq(X, y_noisy, rcond=None)[0]

# Define a Huber style weight rule.
def huber_weights(residuals, delta):
    abs_r = np.abs(residuals)

    return np.where(abs_r <= delta, 1.0, delta / abs_r)

# Start robust fitting values.
robust_coef = ols_coef.copy()
delta = 2.0

# Run a few reweighting steps.
for _ in range(8):
    residuals = y_noisy - X @ robust_coef
    weights = huber_weights(residuals, delta)

    W = np.sqrt(weights)[:, None]
    robust_coef = np.linalg.lstsq(W * X, W[:, 0] * y_noisy, rcond=None)[0]

# Predict lines for plotting.
y_ols = X @ ols_coef
y_robust = X @ robust_coef

# Summarize the fitted slopes.
print(f"OLS slope: {ols_coef[0]:.2f}")
print(f"Robust slope: {robust_coef[0]:.2f}")
print("Robust loss softens extreme residual influence.")

# Plot data and fitted lines.
plt.figure(figsize=(7, 4.5))
plt.scatter(x, y_noisy, color="gray", label="Data")
plt.scatter(x[outlier_idx], y_noisy[outlier_idx], color="crimson", label="Outliers")
plt.plot(x, y_ols, color="royalblue", label="Least squares")

# Add the robust fitted line.
plt.plot(x, y_robust, color="darkorange", label="Robust fit")
plt.xlabel("Distance in miles")
plt.ylabel("Cost in dollars")
plt.title("Robust loss functions resist outliers")

# Finish the single teaching plot.
plt.legend()
plt.tight_layout()
plt.show()



### **3.3. Handling Outliers**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_10/Lecture_A/image_03_03.jpg?v=1783841148" width="250">



>* Outliers distort fits but are not errors.
>* Remove mistakes; adapt models for valid extremes.

>* Robust methods reduce extreme points' influence.
>* Check residuals, leverage, and data context.

>* Outlier handling depends on analysis goals.
>* Robust regression balances stability with real variation.



# <font color="#418FDE" size="6.5" uppercase>**Sparse Robust Models**</font>


In this lecture, you learned to:
- Select sparse linear methods when only a subset of features is expected to matter. 
- Apply Bayesian linear models to estimate coefficients with uncertainty-aware regularization. 
- Use robust regression methods when outliers or nonstandard error patterns affect least squares. 

In the next Lecture (Lecture B), we will go over 'Generalized Bases'